# Paso 1: Generación de Datos Sintéticos

## ¿Qué son los datos sintéticos?

Los **datos sintéticos** son datos generados artificialmente por ordenador que imitan las características de datos reales. En lugar de utilizar información de pacientes reales (lo cual requeriría permisos y plantearía problemas de privacidad), creamos un conjunto de datos ficticio que sigue los mismos patrones estadísticos que encontraríamos en una clínica real.

## ¿Por qué los usamos aquí?

- **Privacidad**: No necesitamos datos de pacientes reales para desarrollar y probar el sistema.
- **Control**: Podemos definir exactamente cuántos deportistas queremos y qué características tendrán.
- **Reproducibilidad**: Con la misma semilla aleatoria, siempre obtenemos exactamente los mismos datos.
- **Desarrollo**: Nos permiten construir y validar el modelo antes de aplicarlo con datos reales.

Este notebook genera el conjunto de datos base que usaremos en todos los pasos siguientes.

In [ ]:
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path
proyecto_raiz = Path("..").resolve()
sys.path.insert(0, str(proyecto_raiz))

from src.generador_datos import generar_dataset
from src.variables import VARIABLES, TOTAL_COLUMNAS
import pandas as pd

## Configuración

Aquí puedes ajustar dos parámetros:

| Parámetro | Descripción | Valor por defecto |
|-----------|-------------|-------------------|
| `N_DEPORTISTAS` | Número de deportistas que tendrá el dataset | 500 |
| `SEMILLA` | Número que controla la aleatoriedad (mismo número = mismos datos siempre) | 42 |

**Recomendación para Roberto**: Empieza con los valores por defecto. Una vez que el sistema funcione correctamente, puedes aumentar `N_DEPORTISTAS` para tener más datos de entrenamiento.

In [ ]:
# TODO ROBERTO: Puedes cambiar el número de deportistas y la semilla
N_DEPORTISTAS = 500
SEMILLA = 42

df = generar_dataset(n_deportistas=N_DEPORTISTAS, semilla=SEMILLA)
print(f"Dataset generado: {df.shape[0]} deportistas, {df.shape[1]} columnas")

## Vista previa de los datos

A continuación se muestran los primeros 10 deportistas del dataset. Puedes ver todas las variables que se han generado para cada uno.

In [ ]:
df.head(10)

## Distribución de niveles de riesgo

Una de las columnas más importantes del dataset es el **nivel de riesgo de lesión**. Aquí podemos ver cuántos deportistas caen en cada categoría.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Tabla de frecuencias
print("Distribución de niveles de riesgo:")
print("="*40)
distribucion = df["riesgo_lesion"].value_counts().sort_index()
for nivel, cantidad in distribucion.items():
    porcentaje = cantidad / len(df) * 100
    print(f"  {nivel}: {cantidad} deportistas ({porcentaje:.1f}%)")
print("="*40)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 5))

colores = {"Bajo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}
niveles = distribucion.index.tolist()
valores = distribucion.values.tolist()
barras_colores = [colores.get(n, "#3498db") for n in niveles]

barras = ax.bar(niveles, valores, color=barras_colores, edgecolor="white", linewidth=1.5)

# Etiquetas encima de cada barra
for barra, valor in zip(barras, valores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 5,
        str(valor),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title("Distribución de niveles de riesgo de lesión", fontsize=14, pad=15)
ax.set_xlabel("Nivel de riesgo", fontsize=12)
ax.set_ylabel("Número de deportistas", fontsize=12)
ax.set_ylim(0, max(valores) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
ruta_fig = proyecto_raiz / "figuras" / "distribucion_riesgo.png"
plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Figura guardada en: {ruta_fig}")

## Estadísticas descriptivas

La tabla siguiente muestra un resumen estadístico de todas las variables numéricas del dataset:

- **count**: número de valores disponibles
- **mean**: media (promedio)
- **std**: desviación estándar (dispersión de los datos)
- **min / max**: valores mínimo y máximo
- **25% / 50% / 75%**: percentiles (el 50% es la mediana)

In [ ]:
df.describe().round(2)

## Guardar datos

Ahora vamos a guardar el dataset generado en un archivo CSV. Este archivo será leído automáticamente por los siguientes notebooks, por lo que es importante ejecutar este paso correctamente.

El archivo se guardará en la carpeta `datos/sinteticos/` dentro del proyecto.

In [ ]:
ruta_salida = proyecto_raiz / "datos" / "sinteticos" / "dataset_sintetico.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Datos guardados en: {ruta_salida}")

## Resumen y siguiente paso

---

**Lo que hemos hecho en este notebook:**

- Generado un dataset sintético con **500 deportistas** y sus variables fisiológicas y de entrenamiento.
- Comprobado que los datos tienen una distribución realista de niveles de riesgo.
- Guardado el dataset en `datos/sinteticos/dataset_sintetico.csv`.

---

**Siguiente paso: ejecuta el notebook `02_exploracion_datos.ipynb`**

En ese notebook exploraremos el dataset en detalle: veremos qué variables están más relacionadas con el riesgo de lesión, detectaremos valores atípicos y prepararemos los datos para el modelo de machine learning.

---

> Si has modificado `N_DEPORTISTAS` o `SEMILLA`, recuerda volver a ejecutar todos los notebooks desde el principio para que los cambios se propaguen correctamente.